# 否定・モダリティ・時制を取り出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_semantics/negation_modality_ja.ipynb)

NLP クックブック 16-11 ／ 意味解析

「行かなかったかもしれない」が、肯定なのか否定なのか、事実なのか推量なのかを判定したい。情報抽出で「起きていないこと」を事実として拾わないために。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import re
from fugashi import Tagger

tagger = Tagger()

MODALITY = [  # (ラベル, 正規表現)。長い表現を先に照合する
    ("推量", r"かもしれない|かもしれません|かも知れない|だろう|でしょう|と思われる|らしい|ようだ|みたいだ|はずだ"),
    ("伝聞", r"そうだ$|そうです$|とのことだ|という$|といいます$|と言われている"),
    ("義務", r"なければならない|なければなりません|べきだ|べきです|ねばならない|ないといけない"),
    ("許可", r"てもよい|てもいい|ても構わない|てもかまわない"),
    ("禁止", r"てはいけない|てはならない|てはいけません|べきではない"),
    ("願望", r"たい$|たいです$|たかった$|てほしい|てほしかった"),
    ("意志", r"つもりだ|つもりです|よう$|ましょう$|ことにする|ことにした"),
    ("依頼", r"てください|てくれ$|ていただけますか|てもらえますか"),
    ("疑問", r"か$|かな$|かしら$|だろうか$|でしょうか$"),
]

def analyze(sentence):
    s = sentence.rstrip("。．！？!?")
    past = any(w.feature.pos1 == "助動詞" and w.feature.lemma == "た" for w in tagger(s))
    modality = [label for label, pat in MODALITY if re.search(pat, s)]
    core = s
    for label, pat in MODALITY:                       # モダリティ表現を除いた「命題」部分で否定を数える
        core = re.sub(pat, "", core)
    negations = sum(1 for w in tagger(core)
                    if (w.feature.pos1 == "助動詞" and w.feature.lemma in ("ない", "ぬ", "ず", "ん", "まい"))
                    or (w.feature.pos1 == "形容詞" and w.feature.lemma == "無い"))
    polarity = "否定" if negations % 2 == 1 else ("肯定 (二重否定)" if negations else "肯定")
    return {"極性": polarity, "時制": "過去" if past else "非過去", "モダリティ": modality or ["断定"]}

for s in ["彼は会議に出席した。", "彼は会議に出席しなかった。", "彼は会議に出席しないかもしれない。", "明日は雨が降るらしい。",
          "レポートを提出しなければならない。", "ここで写真を撮ってはいけない。", "行かないわけにはいかなかった。", "一度も失敗しなかった人はいないだろう。",
          "もっと早く相談してほしかった。", "本当に彼がやったのだろうか。"]:
    print(s, analyze(s))

彼は会議に出席した。 {'極性': '肯定', '時制': '過去', 'モダリティ': ['断定']}
彼は会議に出席しなかった。 {'極性': '否定', '時制': '過去', 'モダリティ': ['断定']}
彼は会議に出席しないかもしれない。 {'極性': '否定', '時制': '非過去', 'モダリティ': ['推量']}
明日は雨が降るらしい。 {'極性': '肯定', '時制': '非過去', 'モダリティ': ['推量']}
レポートを提出しなければならない。 {'極性': '肯定', '時制': '非過去', 'モダリティ': ['義務']}
ここで写真を撮ってはいけない。 {'極性': '肯定', '時制': '非過去', 'モダリティ': ['禁止']}
行かないわけにはいかなかった。 {'極性': '肯定 (二重否定)', '時制': '過去', 'モダリティ': ['断定']}
一度も失敗しなかった人はいないだろう。 {'極性': '肯定 (二重否定)', '時制': '過去', 'モダリティ': ['推量']}
もっと早く相談してほしかった。 {'極性': '肯定', '時制': '過去', 'モダリティ': ['願望']}
本当に彼がやったのだろうか。 {'極性': '肯定', '時制': '過去', 'モダリティ': ['推量', '疑問']}


- 「なければならない」「てはいけない」の「ない」は、否定ではなく義務・禁止の一部。先にモダリティ表現を取り除いてから否定を数えるのがポイント。
- この規則は文末の述語しか見ていない。「来なかった人が合格した」のような従属節の否定、「全員は来なかった」のような否定の作用域 (スコープ) は扱えない。
- 事実性 (その出来事が実際に起きたと書き手が述べているか) の判定は、否定・モダリティ・時制・条件節 (「もし〜なら」) の組み合わせで決まる。医療文書の所見抽出 (「〜を認めない」) などで実用上とても重要。
- この章の前半で扱った KWJA も、基本句の素性として時制やモダリティの情報を出力する。